# Notebook 2 — My own analysis: 1k PBMCs (10x, v3 chemistry)

This is the **independent** run: the same single-cell workflow as Notebook 1, but on a
*different* dataset — ~1,000 PBMCs from a different healthy donor, sequenced with 10x
**v3** chemistry. Different donor, different chemistry, different cell count, so the
numbers and clusters will be my own — not a copy of the tutorial.

Because v3 data is deeper, expect **more genes and counts per cell** than the 3k set —
so I'll re-check the QC thresholds rather than reuse them blindly.

Data source: 10x Genomics, `pbmc_1k_v3` (freely available demo data).
If a cell errors about `leidenalg`/`igraph`, run: `pip install leidenalg igraph`

## 0. Setup

In [ ]:
import os
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '.matplotlib'))
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)

import scanpy as sc
import numpy as np
import pandas as pd
import urllib.request

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')
sc.settings.figdir = 'results'
print('scanpy', sc.__version__)

## 1. Download + load the data

The cell below downloads the 10x `.h5` file (~10 MB) into a local `data/` folder the
first time, then loads it. If the automatic download ever fails, download the file
manually from the 10x website and drop it in the `data/` folder with the same name.

In [ ]:
os.makedirs('data', exist_ok=True)
url  = 'https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_filtered_feature_bc_matrix.h5'
path = 'data/pbmc_1k_v3_filtered_feature_bc_matrix.h5'
if not os.path.exists(path):
    print('downloading 10x PBMC 1k v3 data ...')
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response, open(path, 'wb') as handle:
        handle.write(response.read())
print('file ready:', path)

In [ ]:
adata = sc.read_10x_h5(path)   # cells x genes, raw UMI counts
adata.var_names_make_unique()
adata

## 2. Quality control

Same three metrics as before: genes per cell, total counts, and % mitochondrial reads.

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe()

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

### YOUR TASK - set thresholds for THIS dataset

The 3k tutorial used `n_genes_by_counts < 2500`. This v3 data is deeper, so healthy
cells often have **more** genes - look at your violin plot and pick a sensible upper
cut-off (it may be ~5000-6000 here). Adjust the values below and say why.

_My choice & reasoning:_ I kept `min_genes=200`, used `n_genes_by_counts < 6000`,
and filtered cells with `pct_counts_mt < 15`. The median cell has about 1,919 genes
and the 95th percentile is about 3,740 genes, so 6,000 only removes the most extreme
high-complexity cells that are plausible doublets. The mitochondrial distribution has
a long high-mitochondrial tail, so 15% keeps the main cell population while removing
stressed or damaged cells.

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs.n_genes_by_counts < 6000, :]   # <-- adjust after looking at the plot
adata = adata[adata.obs.pct_counts_mt < 15, :].copy()  # <-- v3 tolerates a higher mito cut; check your plot
print(adata.n_obs, 'cells remain')

## 3. Normalisation

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

## 4. Highly variable genes

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)
adata = adata[:, adata.var.highly_variable].copy()

## 5. Scale + PCA

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack', random_state=0)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

## 6. Neighbours + UMAP

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.pl.umap(adata)

## 7. Clustering (Leiden)

### YOUR TASK - pick a resolution
Try 0.5 / 1.0 / 1.5 and keep the one whose clusters look cleanest. Note your choice.

_My choice & reasoning:_ I tested 0.5, 1.0 and 1.5. Resolution 0.5 merged several
immune populations, while 1.5 split the map into more small clusters than were easy to
interpret for this compact dataset. I kept `resolution=1.0`, which produced 13 clusters
and separated the major PBMC groups plus rare pDC/plasma-cell clusters.

In [ ]:
sc.tl.leiden(
    adata,
    resolution=1.0,
    random_state=0,
    flavor='igraph',
    n_iterations=2,
    directed=False,
)
sc.pl.umap(adata, color=['leiden'], legend_loc='on data', save='_pbmc1k_leiden_1.0.png')

## 8. Marker genes per cluster

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

## 9. Cell-type annotation

Same known PBMC markers as Notebook 1:

| Cell type | Markers |
|---|---|
| CD4 T | IL7R, CD3D |
| CD8 T | CD8A, CD3D |
| B | MS4A1, CD79A |
| NK | GNLY, NKG7 |
| CD14+ Mono | CD14, LYZ |
| FCGR3A+ Mono | FCGR3A, MS4A7 |
| DC | FCER1A, CST3 |
| Platelet | PPBP |

In [ ]:
markers = ['IL7R','CD3D','CD8A','MS4A1','CD79A','GNLY','NKG7',
           'CD14','LYZ','FCGR3A','MS4A7','FCER1A','CST3','PPBP']
sc.pl.dotplot(adata, markers, groupby='leiden', save='_pbmc1k_marker_dotplot.png')

### YOUR TASK - name the clusters

Marker evidence used for the labels below:
- `CD14`, `LYZ`, `S100A8/S100A9` -> CD14+ monocytes
- `MS4A1`, `CD79A`, `CD79B` -> B cells
- `IL7R`, `CD3D`, `CD3E` -> CD4 T cells
- `CD8A`, `CCL5`, `NKG7` -> CD8/cytotoxic T cells
- `GNLY`, `NKG7`, `KLRD1` -> NK cells
- `FCGR3A`, `MS4A7`, `LST1` -> FCGR3A+ monocytes
- `CST3`, `FCER1A` or `IL3RA/LILRA4/TCF4` -> dendritic / plasmacytoid dendritic cells
- `JCHAIN`, `MZB1`, `TNFRSF17` -> plasma cells

I kept duplicate broad populations as separate, uniquely named categories because they
represent substructure within the same major cell type.

In [ ]:
new_names = [
    'CD14+ monocytes I',
    'Naive B cells',
    'CD8 T cells',
    'CD4 T cells',
    'B cells II',
    'CD14+ monocytes II',
    'CD4 T cells / ribosomal-high',
    'Cytotoxic T cells',
    'NK cells',
    'FCGR3A+ monocytes',
    'CST3+ dendritic cells',
    'Plasmacytoid DCs',
    'Plasma cells',
]
adata.rename_categories('leiden', new_names)
adata.obs['cell_type'] = adata.obs['leiden']

major_type = adata.obs['cell_type'].astype(str).replace({
    'CD14+ monocytes I': 'CD14+ monocytes',
    'CD14+ monocytes II': 'CD14+ monocytes',
    'Naive B cells': 'B cells',
    'B cells II': 'B cells',
    'CD4 T cells / ribosomal-high': 'CD4 T cells',
    'CST3+ dendritic cells': 'Dendritic cells',
})
adata.obs['major_cell_type'] = pd.Categorical(major_type)

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)
sc.pl.umap(
    adata,
    color='major_cell_type',
    legend_loc='right margin',
    title='PBMC 1k v3 cell types',
    save='_pbmc1k_cell_types.png',
)

import shutil
shutil.copyfile(
    os.path.join('results', 'umap_pbmc1k_cell_types.png'),
    os.path.join('figures', 'pbmc1k_v3_umap_cell_types.png'),
)

## 10. Save & reflect

The 1k v3 analysis retained 1,055 cells after QC and produced 13 Leiden clusters at
`resolution=1.0`. The marker genes recovered the expected PBMC populations: CD4/CD8 T
cells, cytotoxic T cells, NK cells, B/plasma cells, CD14+ and FCGR3A+ monocytes, plus
small dendritic and plasmacytoid dendritic cell clusters. Compared with the 3k v2
walkthrough, this dataset has fewer cells but deeper v3 capture, so I used a higher
upper gene threshold and a higher mitochondrial cutoff after inspecting the QC plots.

In [ ]:
os.makedirs('results', exist_ok=True)
adata.write('results/pbmc1k_v3_processed.h5ad')
print('saved -> results/pbmc1k_v3_processed.h5ad')
print()
print('Major cell-type counts:')
print(adata.obs['major_cell_type'].value_counts().to_string())
print()
print('Detailed cluster-label counts:')
print(adata.obs['cell_type'].value_counts().to_string())